# 🎯 Aula 15 — Otimização de Processos com IA

**Disciplina:** Inteligência Artificial Aplicada à Engenharia Química  
**Dataset:** `cstr_rendimento.csv` — CSTR com rendimento (T, P)

---

## Contexto

Modelos predizem variáveis de processo. Mas **para que servem depois de prontos**? Para **otimizar**: encontrar as condições (T, P) que maximizam o rendimento — sem testar 10.000 combinações na planta real.

## Metamodelo (Surrogate Model)

Modelo de ML treinado para **aproximar** a resposta do processo. Usado no lugar da planta real para avaliação rápida.

$$\min_{T, P} \; -\hat{y}(T, P) \quad \text{s.a.} \quad T_{min} \leq T \leq T_{max},\; P_{min} \leq P \leq P_{max}$$

## 3.1 — Exercício Guiado: Metamodelo + Otimizador

Treine um Random Forest e encontre T e P que maximizam o rendimento.

### Passo 1: Carregar dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from scipy.optimize import differential_evolution
from sklearn.metrics import r2_score, mean_squared_error

URL = "https://raw.githubusercontent.com/LuisGSVasconcelos/IA_EngQuimica/main/dados/aula15/cstr_rendimento.csv"
df = pd.read_csv(URL)
print(df.head())
print(f"Rendimento: média={df['rendimento_pct'].mean():.1f}%  max={df['rendimento_pct'].max():.1f}%")

### Passo 2: Treinar metamodelo (Random Forest)

In [ ]:
X = df[['T_reator_C', 'pressao_bar']]
y = df['rendimento_pct']

metamodelo = RandomForestRegressor(n_estimators=200, random_state=42)
metamodelo.fit(X, y)
print(f"Metamodelo RF R²: {r2_score(y, metamodelo.predict(X)):.3f}")

### Passo 3: Definir função objetivo

In [ ]:
# Minimizamos -rendimento (para maximizar)
def objetivo(vars_):
    T, P = vars_
    return -metamodelo.predict([[T, P]])[0]

### Passo 4: Otimizar com differential_evolution

In [ ]:
limites = [(70, 130), (5, 12)]   # T, P
res = differential_evolution(objetivo, bounds=limites, seed=42)
T_opt, P_opt = res.x
rend_opt = -res.fun
print(f"Ótimo encontrado: T={T_opt:.1f}°C,  P={P_opt:.1f} bar,  rend={rend_opt:.1f}%")
print(f"(Gabarito: T≈115, P≈8, rend≈92%)")


### Passo 5: Plotar superfície 3D + ponto ótimo

In [ ]:
T_grid = np.linspace(70, 130, 60)
P_grid = np.linspace(5, 12, 60)
TT, PP = np.meshgrid(T_grid, P_grid)
XX = np.column_stack([TT.ravel(), PP.ravel()])
YY = metamodelo.predict(XX).reshape(TT.shape)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(TT, PP, YY, cmap='viridis', alpha=0.8)
ax.scatter([T_opt], [P_opt], [rend_opt], color='red', s=120, label='Ótimo encontrado')
ax.set_xlabel('T (°C)'); ax.set_ylabel('P (bar)'); ax.set_zlabel('Rendimento (%)')
ax.set_title(f'Superfície do metamodelo — Ótimo em T={T_opt:.0f}, P={P_opt:.0f}')
ax.legend()
plt.tight_layout()
plt.show()

### ✏️ Pausa reflexiva (2 min)

O `differential_evolution` acha o mesmo ponto que `minimize`? E se a superfície tiver vários máximos locais?

> _Escreva aqui..._

---

## 3.2 — Exercício em Grupo: Comparação de Métodos

Cada grupo testa um otimizador diferente e compara convergência.

| Grupo | Método | Descrição |
|-------|--------|-----------|
| **A** | `Nelder-Mead` | Simplex, sem gradiente |
| **B** | `BFGS` | Gradiente, ótimo local |
| **C** | `differential_evolution` | Global, evolucionário |
| **D** | `SHGO` | Global, simplex homogêneo |

In [ ]:
from scipy.optimize import minimize, differential_evolution, shgo

# Teste o método do seu grupo
# A: resulta1 = minimize(objetivo, x0=[100, 8], method='Nelder-Mead')
# B: resulta1 = minimize(objetivo, x0=[100, 8], method='BFGS')
# C: resulta1 = differential_evolution(objetivo, bounds=limites, seed=42)
# D: resulta1 = shgo(objetivo, bounds=limites)

# Exemplo (descomente o seu):
resultado = differential_evolution(objetivo, bounds=limites, seed=42)
print(f"Método: differential_evolution")
print(f"Ótimo: T={resultado.x[0]:.1f}, P={resultado.x[1]:.1f}, rend={-resultado.fun:.1f}%")

> **Perguntas:**
> 1. Qual método achou o melhor ótimo?
> 2. Por que BFGS para em ótimo local?
> 3. Quando o BFGS seria preferível mesmo arriscando ótimo local?

### 🧠 Desafio extra (NT)

O ótimo encontrado pelo metamodelo é confiável? Se o modelo erra 2%, o ótimo real pode estar em T diferente. Como mitigar?

> _Escreva aqui..._

---

## Checklist

- [ ] Metamodelo treinado
- [ ] Função objetivo definida
- [ ] Restrições/limites definidos
- [ ] Otimização executada
- [ ] Ótimo validado (superfície + modelo)
- [ ] Gráfico gerado